# Solution: Zero-Shot Forecasting with a Foundation Model

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape

In [ ]:
DATA_PATH = "../data/call_volume.csv"

Run zero-shot Chronos on call volume and evaluate it with historical forecasts. Compare the error metrics against the train/test split baseline.

In [ ]:
# Load data and build TimeSeries
df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").asfreq("MS")
series = TimeSeries.from_series(df["calls"])
train, test = series.split_after(pd.Timestamp("2023-12-01"))

In [ ]:
# Zero-shot forecast for the test period
try:
    from darts.models import Chronos2Model
    chronos = Chronos2Model(model_name="amazon/chronos-bolt-small")
    pred = chronos.predict(len(test), series=train, num_samples=100)
except ImportError:
    model = DartsARIMA(p=2, d=1, q=1)
    model.fit(train)
    pred = model.predict(len(test))

In [ ]:
# Run historical forecasts with a stride of 3 months
try:
    backtest = chronos.historical_forecasts(series, start=0.75, forecast_horizon=12, stride=3)
except:
    model = DartsARIMA(p=2, d=1, q=1)
    model.fit(train)
    backtest = model.historical_forecasts(series, start=0.75, forecast_horizon=12, stride=3)

In [ ]:
# Compute test-set errors
test_pd = test.pd_series()
pred_pd = pred.pd_series()
mae_val = mae(test, pred)
rmse_val = rmse(test, pred)
mape_val = mape(test, pred)
print(f"Chronos MAE: {mae_val:.1f}")
print(f"Chronos RMSE: {rmse_val:.1f}")
print(f"Chronos MAPE: {mape_val:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
series.plot(ax=ax, label="Full Series", color=UN["Black"])
backtest.plot(ax=ax, label="Chronos Backtest", color=US["Orange"])
pred.plot(ax=ax, label="Chronos Zero-Shot", color=UB["Brand Blue"])
ax.axvline(train.time_index[-1], color=UN["Dark Gray"], linestyle="--")
ax.set_title("Chronos Zero-Shot vs Backtest", fontsize=18, fontweight="bold")
ax.set_ylabel("Calls", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend()
plt.tight_layout()
plt.show()